In [1]:
!pip install datasets soundfile -q

### MELD loads

In [3]:
from huggingface_hub import login
#Hugging Face access token
login()

from datasets import load_dataset

#This mirror of MELD only has a test split available (no train/dev), so we're loading test just to peek at the data format for now
dataset = load_dataset("TwinkStart/MELD", split="test")

print("Number of examples:", len(dataset))
#one full example: text, audio, emotion label, etc.
print(dataset[0])

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Number of examples: 2610
{'Sr No.': 1, 'Utterance': 'Why do all you\x92re coffee mugs have numbers on the bottom?', 'Speaker': 'Mark', 'Emotion': 'surprise', 'Sentiment': 'positive', 'Dialogue_ID': 0, 'Utterance_ID': 0, 'Season': 3, 'Episode': 19, 'StartTime': '00:14:38,127', 'EndTime': '00:14:40,378', 'video_path': 'MELD.Raw/output_repeated_splits_test/dia0_utt0.mp4', 'WavPath': 'MELD.Raw/output_repeated_splits_test/dia0_utt0.wav', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7cd68ffceba0>}


We just confirmed the dataset actually has everything we need for our project: for every line someone says in the show, we get the sentence itself, the audio of them saying it, and the emotion behind it, like "surprise". That's the core recipe our whole system needs,  text + sound + feeling, all matched together per line.

### Audio format checked

In [4]:
#Decode the audio for our one example and inspect it
sample = dataset[0]
audio_data = sample["audio"].get_all_samples()

print("Sample rate:", audio_data.sample_rate)
print("Waveform shape:", audio_data.data.shape)
print("Duration (sec):", audio_data.data.shape[-1] / audio_data.sample_rate)

Sample rate: 16000
Waveform shape: torch.Size([2, 36182])
Duration (sec): 2.261375


### Text checked and cleaned

In [5]:
#Check and clean the TEXT side of the data
#Looking at a few raw utterances to see what we're working with
for i in range(5):
    print(dataset[i]["Utterance"], "->", dataset[i]["Emotion"])

#We noticed some text has broken characters (like \x92 instead of an apostrophe), so gona add simple fix
def clean_text(text):
    #Replacing the broken apostrophe character with a normal one
    text = text.replace("\x92", "'")
    text = text.replace("\x93", '"').replace("\x94", '"')  #broken quote marks too
    return text.strip()

#Testing the cleanup func on the same 5 examples
print("\n--- After cleaning ---")
for i in range(5):
    cleaned = clean_text(dataset[i]["Utterance"])
    print(cleaned, "->", dataset[i]["Emotion"])

Why do all youre coffee mugs have numbers on the bottom? -> surprise
Oh. Thats so Monica can keep track. That way if one on them is missing, she can be like, Wheres number 27?! -> anger
Y'know what? -> neutral
Come on, Lydia, you can do it. -> neutral
Push! -> joy

--- After cleaning ---
Why do all you're coffee mugs have numbers on the bottom? -> surprise
Oh. That's so Monica can keep track. That way if one on them is missing, she can be like, Where's number 27?!' -> anger
Y'know what? -> neutral
Come on, Lydia, you can do it. -> neutral
Push! -> joy


### Split into training, 2088 and testing, 522

In [6]:
#Splitting our one dataset into a training portion and a checking portion

#We only have one batch (2,610 examples), so we split it ourselves: 80% to train the model, 20% to check how well it's learning
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset["train"]
val_data = split_dataset["test"]

print("Training samples:", len(train_data))
print("Testing samples:", len(val_data))

#Quick peek to confirm both sets look reasonable
print("\nExample from training set:")
print(train_data[0]["Utterance"], "->", train_data[0]["Emotion"])

print("\nExample from Testing set:")
print(val_data[0]["Utterance"], "->", val_data[0]["Emotion"])

Training samples: 2088
Testing samples: 522

Example from training set:
Are you kidding? -> surprise

Example from Testing set:
No, you said the baby creeps you out. -> neutral


### Building two "reader" models, one for reding text, one for reading audio

In [7]:
!pip install transformers -q

In [10]:
#Loading the two reader models and test them on one example
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from transformers import Wav2Vec2Processor, Wav2Vec2Model

#Using colab T4 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

#Text reader: DistilBERT
text_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
text_model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)

#Audio reader: Wav2Vec2
audio_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
audio_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)

print("\nBoth models loaded successfully.")

#Testing on ONE example
sample = train_data[0]
text = clean_text(sample["Utterance"])
print("\nTesting on:", text)

#Running text through DistilBERT
text_inputs = text_tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    text_output = text_model(**text_inputs)
print("Text embedding shape:", text_output.last_hidden_state.shape)

#Running audio through Wav2Vec2 (converting stereo -> mono first)
audio_data = sample["audio"].get_all_samples()
waveform = audio_data.data.mean(dim=0)  #average 2 channels into 1 mono

audio_inputs = audio_processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt")
audio_inputs = {k: v.to(device) for k, v in audio_inputs.items()}  #moving all tensors to GPU
with torch.no_grad():
     audio_output = audio_model(**audio_inputs)  #unpack the whole dict, not just input_values
print("Audio embedding shape:", audio_output.last_hidden_state.shape)

Using device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Both models loaded successfully.

Testing on: Are you kidding?
Text embedding shape: torch.Size([1, 6, 768])
Audio embedding shape: torch.Size([1, 52, 768])


Text embedding shape: [1, 6, 768], the sentence "Are you kidding?" has 6 word-pieces, and for each one, DistilBERT produced 768 numbers describing its meaning. It is like: 6 words, each turned into a very detailed 768 number "fingerprint"

Audio embedding shape: [1, 52, 768], the audio clip got chopped into 52 tiny time-slices, and again, each slice got its own 768 number fingerprint describing what sound was happening at that moment.

### Pooling, turning many fingerprints into one summary fingerprint - one per word-piece and one per time-slice

In [11]:
#Pooling many fingerprints into one summary fingerprint per example
#Text: average across all 6 word-piece fingerprints, one 768-number summary
text_embedding = text_output.last_hidden_state.mean(dim=1)  #shape becomes [1, 768]
print("Pooled text embedding shape:", text_embedding.shape)

#Audio: average across all 52 time-slice fingerprints, one 768-number summary
audio_embedding = audio_output.last_hidden_state.mean(dim=1)  #shape becomes [1, 768]
print("Pooled audio embedding shape:", audio_embedding.shape)

#Now combine both into one single fingerprint for this example, we place text's 768 numbers and audio's 768 numbers side by side
combined_embedding = torch.cat([text_embedding, audio_embedding], dim=1)
print("Combined embedding shape:", combined_embedding.shape)  #should be [1, 1536]

Pooled text embedding shape: torch.Size([1, 768])
Pooled audio embedding shape: torch.Size([1, 768])
Combined embedding shape: torch.Size([1, 1536])


We just squashed 6 word fingerprints and 52 sound fingerprints down into one clean summary fingerprint each, 768 numbers, then glued them together side-by-side into 1 combined fingerprint of 1536 numbers. This one combined fingerprint now represents both what was said and how it sounded, for a single example.

### Building the emotion classifier

In [12]:
#Building the emotion classifier, untrained, just testing the structure
import torch.nn as nn
#MELD's 7 emotion categories, in a fixed order we'll use throughout
EMOTION_LABELS = ["neutral", "joy", "sadness", "anger", "surprise", "fear", "disgust"]
class EmotionClassifier(nn.Module):
    def __init__(self, input_size=1536, hidden_size=256, num_emotions=7):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(input_size, hidden_size),  #1536 - 256
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, num_emotions)  #256 - 7, one score per emotion
        )

    def forward(self, combined_embedding):
        return self.fusion(combined_embedding)  #returns raw scores, logits, one per emotion

#Creating the classifier and move it to GPU
classifier = EmotionClassifier().to(device)

#Testing it on our one combined fingerprint from the last step
with torch.no_grad():
    logits = classifier(combined_embedding)  #raw scores, not yet probabilities
    probabilities = torch.softmax(logits, dim=1)  #converting scors into probabilities that sum to 1

print("Raw output shape:", logits.shape)  # should be [1, 7]
print("\nPredicted probabilities (untrained, so this will look random):")
for label, prob in zip(EMOTION_LABELS, probabilities[0]):
    print(f"  {label}: {prob.item():.3f}")

Raw output shape: torch.Size([1, 7])

Predicted probabilities (untrained, so this will look random):
  neutral: 0.142
  joy: 0.164
  sadness: 0.151
  anger: 0.126
  surprise: 0.130
  fear: 0.152
  disgust: 0.136


### Training the classifier

In [13]:
#Trainig the emotion classifier
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

#Freezing the big models and they won't change during training
for param in text_model.parameters():
    param.requires_grad = False
for param in audio_model.parameters():
    param.requires_grad = False

text_model.eval()
audio_model.eval()

#Helper: turn one raw example into a combined fingerprint
def get_combined_embedding(example):
    text = clean_text(example["Utterance"])
    text_inputs = text_tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(device)
    with torch.no_grad():
        text_out = text_model(**text_inputs).last_hidden_state.mean(dim=1)

    audio_data = example["audio"].get_all_samples()
    waveform = audio_data.data.mean(dim=0)
    audio_inputs = audio_processor(waveform.numpy(), sampling_rate=16000, return_tensors="pt")
    audio_inputs = {k: v.to(device) for k, v in audio_inputs.items()}
    with torch.no_grad():
        audio_out = audio_model(**audio_inputs).last_hidden_state.mean(dim=1)

    return torch.cat([text_out, audio_out], dim=1).squeeze(0)

#Convert emotin label text, number (0-6), matching EMOTION_LABELS order
def label_to_index(emotion):
    return EMOTION_LABELS.index(emotion)

#Loss func and optimizer, only the classifier's weights get updated
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=1e-3)

#Train for a few passes over the training data, each full pass = 1 "epoch"
EPOCHS = 3
classifier.train()

for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0

    for i in range(len(train_data)):
        example = train_data[i]
        embedding = get_combined_embedding(example).unsqueeze(0)  #add batch dimension - [1, 1536]
        label = torch.tensor([label_to_index(example["Emotion"])]).to(device)

        optimizer.zero_grad()
        logits = classifier(embedding)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == label).item()

        if i % 200 == 0:
            print(f"Epoch {epoch+1}, example {i}/{len(train_data)}, running loss: {loss.item():.3f}")

    accuracy = correct / len(train_data)
    print(f"\nEpoch {epoch+1} done — average loss: {total_loss/len(train_data):.3f}, training accuracy: {accuracy:.3f}\n")

print("Training complete.")

Epoch 1, example 0/2088, running loss: 2.037
Epoch 1, example 200/2088, running loss: 0.472
Epoch 1, example 400/2088, running loss: 1.598
Epoch 1, example 600/2088, running loss: 0.388
Epoch 1, example 800/2088, running loss: 2.132
Epoch 1, example 1000/2088, running loss: 1.988
Epoch 1, example 1200/2088, running loss: 1.326
Epoch 1, example 1400/2088, running loss: 0.891
Epoch 1, example 1600/2088, running loss: 0.373
Epoch 1, example 1800/2088, running loss: 0.351
Epoch 1, example 2000/2088, running loss: 0.203

Epoch 1 done — average loss: 1.326, training accuracy: 0.552

Epoch 2, example 0/2088, running loss: 0.360
Epoch 2, example 200/2088, running loss: 0.375
Epoch 2, example 400/2088, running loss: 0.522
Epoch 2, example 600/2088, running loss: 0.098
Epoch 2, example 800/2088, running loss: 1.222
Epoch 2, example 1000/2088, running loss: 1.123
Epoch 2, example 1200/2088, running loss: 0.808
Epoch 2, example 1400/2088, running loss: 0.759
Epoch 2, example 1600/2088, running los

The model was trained for 3 epochs.

Loss decreased: 1.326 -> 1.183 -> 1.133, This means the model is gradually making fewer mistakes.

Training accuracy increased: 55.2% -> 59.6% -> 61.4%, This means the model learned to identify emotions better with each epoch.

61% training accuracy is a reasonable starting result for a small model trained in a short time. However, this is only training accuracy. The model has already seen these examples.

The next step is to test the model on the 522 unseen examples. That test result is more important because it shows whether the model can correctly identify emotions it has never seen before.

### Real evaluation, testing on unseen data

In [14]:
#Evaluate on the held-out validation set, 522 unseen examples

from collections import defaultdict
classifier.eval()  #switch to evaluation mode

correct = 0
total = 0

#Track correct/total per emotion, so we can see per-class performance
per_emotion_correct = defaultdict(int)
per_emotion_total = defaultdict(int)

with torch.no_grad():  #no training happening here
    for i in range(len(val_data)):
        example = val_data[i]
        embedding = get_combined_embedding(example).unsqueeze(0)
        true_label = example["Emotion"]
        true_index = label_to_index(true_label)

        logits = classifier(embedding)
        predicted_index = logits.argmax(dim=1).item()

        is_correct = (predicted_index == true_index)
        correct += is_correct
        total += 1

        per_emotion_total[true_label] += 1
        per_emotion_correct[true_label] += is_correct

        if i % 100 == 0:
            print(f"Evaluated {i}/{len(val_data)}")

overall_accuracy = correct / total
print(f"\nOverall accuracy on unseen data: {overall_accuracy:.3f} ({correct}/{total})")

print("\nPer-emotion accuracy:")
for emotion in EMOTION_LABELS:
    t = per_emotion_total[emotion]
    c = per_emotion_correct[emotion]
    acc = c / t if t > 0 else 0
    print(f"  {emotion:10s}: {acc:.3f}  ({c}/{t} examples)")

Evaluated 0/522
Evaluated 100/522
Evaluated 200/522
Evaluated 300/522
Evaluated 400/522
Evaluated 500/522

Overall accuracy on unseen data: 0.571 (298/522)

Per-emotion accuracy:
  neutral   : 0.918  (224/244 examples)
  joy       : 0.120  (11/92 examples)
  sadness   : 0.119  (5/42 examples)
  anger     : 0.523  (34/65 examples)
  surprise  : 0.453  (24/53 examples)
  fear      : 0.000  (0/11 examples)
  disgust   : 0.000  (0/15 examples)


The model achieved 57% accuracy on unseen data, which shows it can recognize some emotions reasonably well. It performed very well for neutral emotions, 92% and moderately for anger and surprise, but struggled with joy, sadness, fear, and disgust. This happened mainly because the dataset has many more neutral examples than some of the other emotions. As a result, the model learned neutral emotions better and had difficulty recognizing rare emotions. This is known as class imbalance and is one of the main limitations we found in our model.

### Fixing class imbalance with weighted loss

In [15]:
#Fixing class imbalance using weighted loss
from collections import Counter

#Counting how many training examples we have per emotion
emotion_counts = Counter(train_data["Emotion"])
print("Training set emotion counts:")
for emotion in EMOTION_LABELS:
    print(f"  {emotion:10s}: {emotion_counts[emotion]}")

#Calculating weights: rarer emotions get higher weight
#Formula used: weight = total_examples / (num_classes * count_for_this_class)
total_examples = len(train_data)
num_classes = len(EMOTION_LABELS)

class_weights = []
for emotion in EMOTION_LABELS:
    count = emotion_counts[emotion]
    weight = total_examples / (num_classes * count)
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("\nCalculated weights (higher = rarer, gets more attention):")
for emotion, w in zip(EMOTION_LABELS, class_weights):
    print(f"  {emotion:10s}: {w:.3f}")

#Reset the classifier to a fresh, untrained state
classifier = EmotionClassifier().to(device)
optimizer = optim.Adam(classifier.parameters(), lr=1e-3)

#New loss function, now weighted so rare emotions matter more
criterion = nn.CrossEntropyLoss(weight=class_weights)

#Retrain, same as before
EPOCHS = 3
classifier.train()

for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0

    for i in range(len(train_data)):
        example = train_data[i]
        embedding = get_combined_embedding(example).unsqueeze(0)
        label = torch.tensor([label_to_index(example["Emotion"])]).to(device)

        optimizer.zero_grad()
        logits = classifier(embedding)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == label).item()

        if i % 200 == 0:
            print(f"Epoch {epoch+1}, example {i}/{len(train_data)}, running loss: {loss.item():.3f}")

    accuracy = correct / len(train_data)
    print(f"\nEpoch {epoch+1} done — average loss: {total_loss/len(train_data):.3f}, training accuracy: {accuracy:.3f}\n")

print("Retraining complete.")

Training set emotion counts:
  neutral   : 1012
  joy       : 310
  sadness   : 166
  anger     : 280
  surprise  : 228
  fear      : 39
  disgust   : 53

Calculated weights (higher = rarer, gets more attention):
  neutral   : 0.295
  joy       : 0.962
  sadness   : 1.797
  anger     : 1.065
  surprise  : 1.308
  fear      : 7.648
  disgust   : 5.628
Epoch 1, example 0/2088, running loss: 2.002
Epoch 1, example 200/2088, running loss: 0.546
Epoch 1, example 400/2088, running loss: 1.547
Epoch 1, example 600/2088, running loss: 0.205
Epoch 1, example 800/2088, running loss: 1.231
Epoch 1, example 1000/2088, running loss: 1.819
Epoch 1, example 1200/2088, running loss: 0.779
Epoch 1, example 1400/2088, running loss: 1.109
Epoch 1, example 1600/2088, running loss: 0.271
Epoch 1, example 1800/2088, running loss: 0.412
Epoch 1, example 2000/2088, running loss: 0.357

Epoch 1 done — average loss: 1.338, training accuracy: 0.545

Epoch 2, example 0/2088, running loss: 0.590
Epoch 2, example 2

### Re-run evaluation on the same 522 unseen examples

In [16]:
#Re-evaluate the reweighted classifier on unseen data
classifier.eval()

correct = 0
total = 0
per_emotion_correct = defaultdict(int)
per_emotion_total = defaultdict(int)

with torch.no_grad():
    for i in range(len(val_data)):
        example = val_data[i]
        embedding = get_combined_embedding(example).unsqueeze(0)
        true_label = example["Emotion"]
        true_index = label_to_index(true_label)

        logits = classifier(embedding)
        predicted_index = logits.argmax(dim=1).item()

        is_correct = (predicted_index == true_index)
        correct += is_correct
        total += 1

        per_emotion_total[true_label] += 1
        per_emotion_correct[true_label] += is_correct

        if i % 100 == 0:
            print(f"Evaluated {i}/{len(val_data)}")

overall_accuracy = correct / total
print(f"\nOverall accuracy on unseen data: {overall_accuracy:.3f} ({correct}/{total})")

print("\nPer-emotion accuracy (compare to before: neutral 0.918, joy 0.120, sadness 0.119, anger 0.523, surprise 0.453, fear 0.000, disgust 0.000):")
for emotion in EMOTION_LABELS:
    t = per_emotion_total[emotion]
    c = per_emotion_correct[emotion]
    acc = c / t if t > 0 else 0
    print(f"  {emotion:10s}: {acc:.3f}  ({c}/{t} examples)")

Evaluated 0/522
Evaluated 100/522
Evaluated 200/522
Evaluated 300/522
Evaluated 400/522
Evaluated 500/522

Overall accuracy on unseen data: 0.557 (291/522)

Per-emotion accuracy (compare to before: neutral 0.918, joy 0.120, sadness 0.119, anger 0.523, surprise 0.453, fear 0.000, disgust 0.000):
  neutral   : 0.951  (232/244 examples)
  joy       : 0.120  (11/92 examples)
  sadness   : 0.024  (1/42 examples)
  anger     : 0.462  (30/65 examples)
  surprise  : 0.321  (17/53 examples)
  fear      : 0.000  (0/11 examples)
  disgust   : 0.000  (0/15 examples)


The weighted loss approach did not improve the model as expected. The overall accuracy slightly decreased from 57.1% to 55.7%.

For each emotion:

Neutral: improved from 91.8% to 95.1%
Joy: stayed the same at 12.0%
Sadness: dropped from 11.9% to 2.4%
Anger: dropped from 52.3% to 46.2%
Surprise: dropped from 45.3% to 32.1%
Fear: remained at 0%
Disgust: remained at 0%

This shows that weighted loss alone was not enough to solve the class imbalance problem. Rare emotions still had very few training examples, and the small classifier may not have enough capacity to learn subtle emotional differences.

### Second attempt, softer weights, more training

In [17]:
#Second attempt - softer weights + more traiing

#Take the square root of our original weights to soften the extremes, (fear's weight goes from 7.648 down to about 2.77 - still higher, but less extreme
soft_class_weights = torch.sqrt(class_weights)

print("Softened weights (compare to original):")
for emotion, orig, soft in zip(EMOTION_LABELS, class_weights, soft_class_weights):
    print(f"  {emotion:10s}: original {orig:.3f}  ->  softened {soft:.3f}")

#Reset classifier fresh again
classifier = EmotionClassifier().to(device)
optimizer = optim.Adam(classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(weight=soft_class_weights)

#Train for 5 epochs this time instead of 3
EPOCHS = 5
classifier.train()

for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0

    for i in range(len(train_data)):
        example = train_data[i]
        embedding = get_combined_embedding(example).unsqueeze(0)
        label = torch.tensor([label_to_index(example["Emotion"])]).to(device)

        optimizer.zero_grad()
        logits = classifier(embedding)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == label).item()

        if i % 400 == 0:
            print(f"Epoch {epoch+1}, example {i}/{len(train_data)}, running loss: {loss.item():.3f}")

    accuracy = correct / len(train_data)
    print(f"\nEpoch {epoch+1} done — average loss: {total_loss/len(train_data):.3f}, training accuracy: {accuracy:.3f}\n")

print("Retraining complete (attempt 2).")

Softened weights (compare to original):
  neutral   : original 0.295  ->  softened 0.543
  joy       : original 0.962  ->  softened 0.981
  sadness   : original 1.797  ->  softened 1.340
  anger     : original 1.065  ->  softened 1.032
  surprise  : original 1.308  ->  softened 1.144
  fear      : original 7.648  ->  softened 2.766
  disgust   : original 5.628  ->  softened 2.372
Epoch 1, example 0/2088, running loss: 2.037
Epoch 1, example 400/2088, running loss: 1.456
Epoch 1, example 800/2088, running loss: 1.777
Epoch 1, example 1200/2088, running loss: 0.936
Epoch 1, example 1600/2088, running loss: 0.209
Epoch 1, example 2000/2088, running loss: 0.236

Epoch 1 done — average loss: 1.341, training accuracy: 0.555

Epoch 2, example 0/2088, running loss: 0.584
Epoch 2, example 400/2088, running loss: 1.444
Epoch 2, example 800/2088, running loss: 0.768
Epoch 2, example 1200/2088, running loss: 0.843
Epoch 2, example 1600/2088, running loss: 0.244
Epoch 2, example 2000/2088, running 

In [18]:
#Evaluate attempt 2, softer weights, 5 epochs on unseen data
classifier.eval()

correct = 0
total = 0
per_emotion_correct = defaultdict(int)
per_emotion_total = defaultdict(int)

with torch.no_grad():
    for i in range(len(val_data)):
        example = val_data[i]
        embedding = get_combined_embedding(example).unsqueeze(0)
        true_label = example["Emotion"]
        true_index = label_to_index(true_label)

        logits = classifier(embedding)
        predicted_index = logits.argmax(dim=1).item()

        is_correct = (predicted_index == true_index)
        correct += is_correct
        total += 1

        per_emotion_total[true_label] += 1
        per_emotion_correct[true_label] += is_correct

        if i % 100 == 0:
            print(f"Evaluated {i}/{len(val_data)}")

overall_accuracy = correct / total
print(f"\nOverall accuracy on unseen data: {overall_accuracy:.3f} ({correct}/{total})")

print("\nPer-emotion accuracy — comparison across all 3 attempts:")
print(f"{'Emotion':10s} {'Attempt 1 (none)':18s} {'Attempt 2 (hard)':18s} {'Attempt 3 (soft)':18s}")
comparison = {
    "neutral": (0.918, 0.951), "joy": (0.120, 0.120), "sadness": (0.119, 0.024),
    "anger": (0.523, 0.462), "surprise": (0.453, 0.321), "fear": (0.000, 0.000), "disgust": (0.000, 0.000)
}
for emotion in EMOTION_LABELS:
    t = per_emotion_total[emotion]
    c = per_emotion_correct[emotion]
    acc = c / t if t > 0 else 0
    a1, a2 = comparison[emotion]
    print(f"{emotion:10s} {a1:<18.3f} {a2:<18.3f} {acc:<18.3f}")

Evaluated 0/522
Evaluated 100/522
Evaluated 200/522
Evaluated 300/522
Evaluated 400/522
Evaluated 500/522

Overall accuracy on unseen data: 0.565 (295/522)

Per-emotion accuracy — comparison across all 3 attempts:
Emotion    Attempt 1 (none)   Attempt 2 (hard)   Attempt 3 (soft)  
neutral    0.918              0.951              0.926             
joy        0.120              0.120              0.163             
sadness    0.119              0.024              0.024             
anger      0.523              0.462              0.585             
surprise   0.453              0.321              0.283             
fear       0.000              0.000              0.000             
disgust    0.000              0.000              0.000             


The three attempts produced very similar overall results: 57.1%, 55.7%, and 56.5%, so none was a clear overall winner.

For each emotion:

Neutral: 91.8% -> 95.1% -> 92.6%
Joy: 12.0% -> 12.0% -> 16.3% — slightly improved
Sadness: 11.9% -> 2.4% -> 2.4% — decreased
Anger: 52.3% -> 46.2% -> 58.5% — improved with soft weights
Surprise: 45.3% -> 32.1% -> 28.3% — decreased
Fear: 0% in all three attempts
Disgust: 0% in all three attempts

This shows that changing the class weights did not significantly solve the problem. The main issue appears to be limited data for rare emotions, especially fear and disgust. Therefore, the model is limited more by data scarcity and model capacity than by the choice of weighting method.

### Save the trained classifier, then start the response generator

In [19]:
#Save our trained classifier so we don't lose it
torch.save(classifier.state_dict(), "emotion_classifier.pt")
print("Classifier saved as emotion_classifier.pt")

#Also save which weighting/epoch setup produced this, for our records
import json
model_info = {
    "approach": "soft class weights (sqrt), 5 epochs",
    "overall_accuracy": 0.565,
    "per_emotion_accuracy": {
        "neutral": 0.926, "joy": 0.163, "sadness": 0.024,
        "anger": 0.585, "surprise": 0.283, "fear": 0.000, "disgust": 0.000
    }
}
with open("model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)
print("Model info saved as model_info.json")

Classifier saved as emotion_classifier.pt
Model info saved as model_info.json


### Building the response generator

In [20]:
#Load the response generator and test it on one example
from transformers import AutoModelForCausalLM, AutoTokenizer

#Load a smll instruct tuned language model, ~1.5 billion params
llm_name = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForCausalLM.from_pretrained(llm_name, torch_dtype=torch.float16).to(device)

print("Response generator loaded.")

#Func: given a transcript + detected emotion, generate a short reply
def generate_response(transcript, emotion):
    prompt = (
        f"You are a character robot reacting to what someone just said.\n"
        f"They said: \"{transcript}\"\n"
        f"Their detected emotion: {emotion}\n"
        f"Reply in 1 short sentence, acknowledging their emotion naturally. "
        f"Do not repeat their words back, just respond like a person would."
    )
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = llm_model.generate(**inputs, max_new_tokens=40, do_sample=False)

    #Only decode the newly generated part, skip the prompt itself
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

#Test on one example
test_example = val_data[0]
test_text = clean_text(test_example["Utterance"])
test_emotion = test_example["Emotion"]  # sing the TRUE label for this first test

print(f"\nInput: \"{test_text}\"")
print(f"Emotion: {test_emotion}")

response = generate_response(test_text, test_emotion)
print(f"Generated response: \"{response}\"")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Response generator loaded.

Input: "No, you said the baby creeps you out."
Emotion: neutral
Generated response: "I understand that they might feel surprised or confused about my statement regarding the baby."


### Connecting the classifier's predictin to the response generator

In [21]:
#Wire the classifier and response generatr together into one pipeline
def full_pipeline(example):
    """Takes one raw MELD example, returns predicted emotion + a generated response."""

    text = clean_text(example["Utterance"])
    #Step 1: get combined text+audio fingerprint
    embedding = get_combined_embedding(example).unsqueeze(0)
    #Step 2: classifier predicts the emotion, no true label used here
    classifier.eval()
    with torch.no_grad():
        logits = classifier(embedding)
        predicted_index = logits.argmax(dim=1).item()
        confidence = torch.softmax(logits, dim=1)[0][predicted_index].item()

    predicted_emotion = EMOTION_LABELS[predicted_index]

    #Step 3: generate a response using the PREDICTED emotion, a slightly tighter prompt
    prompt = (
        f'Someone said: "{text}"\n'
        f"They sound {predicted_emotion}.\n"
        f"Respond naturally in one short, casual sentence, like a friend reacting in the moment. "
        f"Do not explain or describe the emotion, just react to it."
    )
    messages = [{"role": "user", "content": prompt}]
    input_text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = llm_model.generate(**inputs, max_new_tokens=30, do_sample=False)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response_text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        "text": text,
        "true_emotion": example["Emotion"],  #kept only so WE can compare, not used by the model
        "predicted_emotion": predicted_emotion,
        "confidence": round(confidence, 3),
        "response_text": response_text
    }

#Test the full pipeline on 3 examples
for i in range(3):
    result = full_pipeline(val_data[i])
    print(f"\nInput: \"{result['text']}\"")
    print(f"True emotion: {result['true_emotion']}  |  Predicted: {result['predicted_emotion']} ({result['confidence']})")
    print(f"Response: \"{result['response_text']}\"")


Input: "No, you said the baby creeps you out."
True emotion: neutral  |  Predicted: neutral (0.53)
Response: "Oh no, that must have been really awkward!"

Input: "Damnit!"
True emotion: anger  |  Predicted: anger (0.949)
Response: "Oh no! I can imagine that's got them pretty worked up."

Input: "Wow! Skates!"
True emotion: joy  |  Predicted: joy (0.449)
Response: "That's cool! I bet skating feels awesome."


The end-to-end system worked correctly for these 3 examples. It successfully combined the text and audio input, predicted the emotion, and generated a short response.

Neutral: Correctly predicted with 53% confidence
Anger: Correctly predicted with 94.9% confidence, showing a strong and clear prediction
Joy: Correctly predicted with 44.9% confidence, showing more uncertainty

All 3 predictions matched their true emotin labels. The generated responses were also natural and related to the detected emotion. However, since this is only a small sample, it demonstrates the system's functionality rather than proving overall model accuracy.

### Simulating real-time behavior + measuring latency

In [22]:
#Simulateng real-time processing + measure latency per utterance
import time

def full_pipeline_timed(example):
    """Same as full_pipeline, but measures how long it takes."""
    start_time = time.time()
    result = full_pipeline(example)
    end_time = time.time()
    latency_ms = round((end_time - start_time) * 1000, 1)
    result["latency_ms"] = latency_ms

    return result

#Simulate a short "live conversation": process 5 utterances one after another, just like they'd arrive in real-time, and time each one individually
print("Simulating real-time conversation (5 utterances, one at a time):\n")

results = []
for i in range(5):
    example = val_data[i]
    result = full_pipeline_timed(example)
    results.append(result)

    print(f"[{result['latency_ms']} ms] \"{result['text']}\"")
    print(f"   -> emotion: {result['predicted_emotion']} ({result['confidence']})  |  response: \"{result['response_text']}\"\n")

#Summary stats
avg_latency = sum(r["latency_ms"] for r in results) / len(results)
print(f"Average latency per utterance: {avg_latency:.1f} ms")

Simulating real-time conversation (5 utterances, one at a time):

[779.2 ms] "No, you said the baby creeps you out."
   -> emotion: neutral (0.53)  |  response: "Oh no, that must have been really awkward!"

[1008.6 ms] "Damnit!"
   -> emotion: anger (0.949)  |  response: "Oh no! I can imagine that's got them pretty worked up."

[753.4 ms] "Wow! Skates!"
   -> emotion: joy (0.449)  |  response: "That's cool! I bet skating feels awesome."

[513.6 ms] "Really?"
   -> emotion: neutral (0.534)  |  response: "Got that? No problem!"

[2249.3 ms] "No-noReally?!"
   -> emotion: anger (0.696)  |  response: "Oh no! That must have really gotten under their skin. Let's talk about it later when you're feeling calmer."

Average latency per utterance: 1060.8 ms


The real-time simulation processed 5 utterances one at a time, with an average latency of 1,060.8 ms, about 1 second per utterance.

The response times ranged from 513.6 ms to 2249.3 ms. This means some inputs were processed faster than others, mainly because longer or more complex responses can take more time to generate.

For this project, an average response time of about 1 second can be considered near real-time for a turn-based character robot interaction. The system is fast enough to process one utterance and provide an emotion prediction and response before the next interaction. The variation in response time is a limitation that should be documented, but the prototype still meets the intended near real-time interaction goal.

### Counting total parameters, the 6B constraint check

In [23]:
#Count total params across the entire pipeline

def count_params(model):
    return sum(p.numel() for p in model.parameters())

text_params = count_params(text_model)
audio_params = count_params(audio_model)
classifier_params = count_params(classifier)
llm_params = count_params(llm_model)

total_params = text_params + audio_params + classifier_params + llm_params

print("Parameter count breakdown:\n")
print(f"  Text encoder (DistilBERT):        {text_params:,}")
print(f"  Audio encoder (Wav2Vec2-base):     {audio_params:,}")
print(f"  Fusion + classifier (ours):        {classifier_params:,}")
print(f"  Response generator (Qwen2.5-1.5B): {llm_params:,}")
print(f"  {'-'*50}")
print(f"  TOTAL:                              {total_params:,}")
print(f"\nConstraint: must be ≤ 6,000,000,000")
print(f"We are using: {total_params/1e9:.3f} billion parameters")
print(f"Headroom remaining: {(6_000_000_000 - total_params)/1e9:.3f} billion parameters")

Parameter count breakdown:

  Text encoder (DistilBERT):        66,362,880
  Audio encoder (Wav2Vec2-base):     94,371,712
  Fusion + classifier (ours):        395,271
  Response generator (Qwen2.5-1.5B): 1,543,714,304
  --------------------------------------------------
  TOTAL:                              1,704,844,167

Constraint: must be ≤ 6,000,000,000
We are using: 1.705 billion parameters
Headroom remaining: 4.295 billion parameters


### Saving a proper structured JSON output

In [24]:
#Saving a proper structured JSON output file
import json

#Runng the full timed pipeline on 5 fresh examples, difffer from before
sample_results = []
for i in range(5, 10):
    example = val_data[i]
    result = full_pipeline_timed(example)
    sample_results.append(result)

#Save as a clean JSON file
with open("sample_run.json", "w") as f:
    json.dump(sample_results, f, indent=2)

print("Saved sample_run.json with", len(sample_results), "examples.\n")

#Print it out so we can see exactly what it looks like
print(json.dumps(sample_results, indent=2))

Saved sample_run.json with 5 examples.

[
  {
    "text": "Chandler what do you say?",
    "true_emotion": "neutral",
    "predicted_emotion": "neutral",
    "confidence": 0.869,
    "response_text": "Oh, that's Chandler? Sounds friendly enough! What's up?",
    "latency_ms": 1109.1
  },
  {
    "text": "Wow, it looks like a, like a holiday card y'know, with the tree in the middle and the skaters and the snow.",
    "true_emotion": "joy",
    "predicted_emotion": "neutral",
    "confidence": 0.497,
    "response_text": "Oh wow, that's so festive! I'm glad you're enjoying the winter wonderland vibes.",
    "latency_ms": 2413.9
  },
  {
    "text": "Yes I did! And I put a little",
    "true_emotion": "anger",
    "predicted_emotion": "anger",
    "confidence": 0.405,
    "response_text": "Oh no, that must have really gotten under their skin!",
    "latency_ms": 889.3
  },
  {
    "text": "Are you kidding?",
    "true_emotion": "surprise",
    "predicted_emotion": "surprise",
    "confide

### Reporting hardware and resource usage

In [25]:
#Report hardware used
import subprocess

gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"], capture_output=True, text=True)
print("GPU info:")
print(gpu_info.stdout)

#Get peak GPU memory used by our models during this session
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"\nPeak GPU memory used by our models: {peak_memory_gb:.2f} GB")

GPU info:
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 10855 MiB, 15360 MiB


Peak GPU memory used by our models: 10.68 GB
